In [3]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.mr.300.vec.gz
!gunzip cc.mr.300.vec.gz

'wget' is not recognized as an internal or external command,
operable program or batch file.
'gunzip' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import os
import requests
import zipfile
import gzip
import shutil
from tqdm import tqdm

def download_file(url, filename):
    if os.path.exists(filename):
        print(f"{filename} already exists.")
        return
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    with open(filename, 'wb') as f, tqdm(total=total_size, unit='iB', unit_scale=True) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = f.write(data)
            bar.update(size)

# 1. Handle FastText (Marathi)
print("Downloading FastText...")
download_file("https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.mr.300.vec.gz", "cc.mr.300.vec.gz")
print("Decompressing FastText...")
with gzip.open('cc.mr.300.vec.gz', 'rb') as f_in:
    with open('cc.mr.300.vec', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

# 2. Handle GloVe
print("\nDownloading GloVe...")
download_file("http://nlp.stanford.edu/data/glove.6B.zip", "glove.6B.zip")
print("Unzipping GloVe...")
with zipfile.ZipFile("glove.6B.zip", "r") as zip_ref:
    zip_ref.extractall(".")

print("\nDone! Files are ready.")

  2%|▏         | 10.5M/505M [09:05<12:29:26, 11.0kiB/s]

In [2]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.5 MB/s eta 0:00:00


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 13.5 MB/s eta 0:00:00


In [8]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2026-05-01 13:41:32--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-05-01 13:41:33--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-05-01 13:41:33--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from tqdm import tqdm
import nltk
from nltk.tokenize import word_tokenize
import matplotlib.pyplot as plt
import sacrebleu
from gensim.models import KeyedVectors

nltk.download('punkt')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [15]:
train_df = pd.read_csv('/content/team19_mr_train.csv')
val_df = pd.read_csv('/content/team19_mr_valid.csv')
test_df = pd.read_csv('/content/team19_mr_test.csv')

train_df.columns = ['source', 'target']
val_df.columns = ['source', 'target']
test_df.columns = ['source', 'target']

In [19]:
# from gensim.models import KeyedVectors
# glove = KeyedVectors.load_word2vec_format('glove.6B.100d.txt', binary=False, no_header=True)
# glove_dim = glove.vector_size
# indic_tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')

# from gensim.models import KeyedVectors
# fasttext = KeyedVectors.load_word2vec_format('cc.hi.300.vec', binary=False)
# fasttext_dim = fasttext.vector_size

In [24]:
# =========================
# LOAD EMBEDDINGS
# =========================
from gensim.models import KeyedVectors

glove = KeyedVectors.load_word2vec_format(
    'glove.6B.100d.txt', binary=False, no_header=True
)
glove_dim = glove.vector_size

from gensim.models import KeyedVectors

fasttext = KeyedVectors.load_word2vec_format(
    'cc.mr.300.vec',
    binary=False,
    limit=200000   # IMPORTANT to avoid RAM crash
)

In [13]:
# all_target_sentences = list(train_df['target']) + list(val_df['target']) + list(test_df['target'])
# all_tokens = set()
# for sentence in all_target_sentences:
#     tokens = indic_tokenizer.tokenize(sentence)[:510]
#     all_tokens.update(tokens)

# special_tokens = ['<pad>', '<sos>', '<eos>', '<unk>']
# token2idx = {token: idx + len(special_tokens) for idx, token in enumerate(sorted(all_tokens))}

# vocab_size = max(token2idx.values()) + 1
# embedding_matrix = np.zeros((vocab_size, fasttext_dim))

# for token, idx in token2idx.items():
#     if token in fasttext:
#         embedding_matrix[idx] = fasttext[token]
#     else:
#         embedding_matrix[idx] = np.random.normal(size=(fasttext_dim,))

# for idx, token in enumerate(special_tokens):
#     token2idx[token] = idx
# idx2token = {v: k for k, v in token2idx.items()}

# with open('token2idx.pkl', 'wb') as f:
#     pickle.dump(token2idx, f)

Token indices sequence length is longer than the specified maximum sequence length for this model (755 > 512). Running this sequence through the model will result in indexing errors


IndexError: index 4310 is out of bounds for axis 0 with size 4310

In [ ]:
# def encode_english_sentence(sentence, max_len=30):
#     tokens = nltk.word_tokenize(sentence.lower())
#     vectors = []
#     for token in tokens:
#         if token in glove:
#             vectors.append(glove[token])
#         else:
#             vectors.append(np.zeros(glove_dim))
#     if len(vectors) < max_len:
#         vectors += [np.zeros(glove_dim)] * (max_len - len(vectors))
#     else:
#         vectors = vectors[:max_len]
#     return np.array(vectors)

# def encode_indic_sentence(sentence, max_len=30):
#     tokens = indic_tokenizer.tokenize(sentence)
#     indices = [token2idx.get('<sos>')] + [token2idx.get(token, token2idx['<unk>']) for token in tokens] + [token2idx.get('<eos>')]
#     if len(indices) < max_len:
#         indices += [token2idx.get('<pad>')] * (max_len - len(indices))
#     else:
#         indices = indices[:max_len]
#     return indices

In [26]:
fasttext_dim = fasttext.vector_size

In [27]:
# =========================
# TOKENIZATION
# =========================
def tokenize_en(text):
    return text.lower().strip().split()

def tokenize_mr(text):
    return text.strip().split()

# =========================
# ENCODING
# =========================
def encode_english(sentence, max_len=30):
    tokens = tokenize_en(sentence)
    vectors = []

    for token in tokens:
        if token in glove:
            vectors.append(glove[token])
        else:
            vectors.append(np.random.normal(size=(glove_dim,)))

    if len(vectors) < max_len:
        vectors += [np.zeros(glove_dim)] * (max_len - len(vectors))
    else:
        vectors = vectors[:max_len]

    return np.array(vectors)


def encode_marathi(sentence, max_len=30):
    tokens = tokenize_mr(sentence)
    vectors = []

    for token in tokens:
        if token in fasttext:
            vectors.append(fasttext[token])
        else:
            vectors.append(np.random.normal(size=(fasttext_dim,)))

    if len(vectors) < max_len:
        vectors += [np.zeros(fasttext_dim)] * (max_len - len(vectors))
    else:
        vectors = vectors[:max_len]

    return np.array(vectors)

In [ ]:

# class TranslationDataset(Dataset):
#     def __init__(self, df):
#         self.sources = df['source'].tolist()
#         self.targets = df['target'].tolist()

#     def __len__(self):
#         return len(self.sources)

#     def __getitem__(self, idx):
#         source = encode_english_sentence(self.sources[idx])
#         target = encode_indic_sentence(self.targets[idx])
#         return torch.tensor(source, dtype=torch.float32), torch.tensor(target, dtype=torch.long)

# train_dataset = TranslationDataset(train_df)
# val_dataset = TranslationDataset(val_df)
# test_dataset = TranslationDataset(test_df)

# train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=10)
# test_loader = DataLoader(test_dataset, batch_size=10)

In [28]:
# =========================
# DATASET
# =========================
class TranslationDataset(Dataset):
    def __init__(self, df):
        self.src = df['source'].tolist()
        self.tgt = df['target'].tolist()

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        src = encode_english(self.src[idx])
        tgt = encode_marathi(self.tgt[idx])

        return (
            torch.tensor(src, dtype=torch.float32),
            torch.tensor(tgt, dtype=torch.float32)
        )

train_loader = DataLoader(TranslationDataset(train_df), batch_size=16, shuffle=True)
val_loader = DataLoader(TranslationDataset(val_df), batch_size=16)
test_loader = DataLoader(TranslationDataset(test_df), batch_size=16)


In [ ]:
# def get_sinusoidal_positional_encoding(seq_len, d_model, device):
#     position = torch.arange(0, seq_len, dtype=torch.float32).unsqueeze(1)
#     div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

#     pe = torch.zeros(seq_len, d_model)
#     pe[:, 0::2] = torch.sin(position * div_term)
#     pe[:, 1::2] = torch.cos(position * div_term)

#     return pe.to(device)

In [29]:
# =========================
# POSITIONAL ENCODING
# =========================
def get_positional_encoding(seq_len, d_model, device):
    pos = torch.arange(seq_len, device=device).unsqueeze(1)
    div = torch.exp(torch.arange(0, d_model, 2, device=device) *
                    (-np.log(10000.0) / d_model))

    pe = torch.zeros(seq_len, d_model, device=device)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)

    return pe

In [30]:
# class TransformerMTModel(nn.Module):
#     def __init__(self, tgt_vocab_size, embed_dim=300, num_heads=10, num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=512, max_len=100):
#         super().__init__()
#         self.embed_dim = embed_dim
#         self.max_len = max_len

#         self.encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=dim_feedforward)
#         self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=num_encoder_layers)
#         self.src_projection = nn.Linear(100, 300)
#         self.tgt_embedding = nn.Embedding.from_pretrained(
#     torch.tensor(embedding_matrix, dtype=torch.float32),
#     freeze=False
# )
#         self.decoder_layer = nn.TransformerDecoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=dim_feedforward)
#         self.decoder = nn.TransformerDecoder(self.decoder_layer, num_layers=num_decoder_layers)

#         self.output_layer = nn.Linear(embed_dim, tgt_vocab_size)

#     def generate_square_subsequent_mask(self, sz):
#         # Generates causal mask for target to mask future tokens
#         mask = torch.triu(torch.ones(sz, sz), diagonal=1).bool()
#         return mask

#     def forward(self, src, tgt):
#         # src: [batch_size, src_len, embed_dim]
#         # tgt: [batch_size, tgt_len]
#         batch_size, src_len, _ = src.size()
#         tgt_len = tgt.size(1)

#         # Apply sinusoidal positional encoding
#         src_pe = get_sinusoidal_positional_encoding(src_len, self.embed_dim, src.device)  # [src_len, embed_dim]
#         tgt_pe = get_sinusoidal_positional_encoding(tgt_len, self.embed_dim, tgt.device)  # [tgt_len, embed_dim]

#         # Add PE (broadcast over batch)
#         src = self.src_projection(src)
#         src = src + src_pe.unsqueeze(0)
#         tgt_emb = self.tgt_embedding(tgt) + tgt_pe.unsqueeze(0)  # [batch_size, tgt_len, embed_dim]

#         # Transformer expects [seq_len, batch_size, embed_dim]
#         src = src.permute(1, 0, 2)
#         tgt_emb = tgt_emb.permute(1, 0, 2)

#         memory = self.encoder(src)
#         # Generate causal mask for decoder
#         tgt_mask = self.generate_square_subsequent_mask(tgt_len).to(tgt.device)

#         # Pass mask to decoder to mask future tokens
#         output = self.decoder(tgt_emb, memory, tgt_mask=tgt_mask)
#         output = self.output_layer(output)
#         return output.permute(1, 0, 2)

# model = TransformerMTModel(
#     tgt_vocab_size=len(token2idx),
#     num_encoder_layers=3,
#     num_decoder_layers=3
# ).to(device)

In [31]:
# =========================
# MODEL
# =========================
class TransformerMT(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=100, nhead=5),
            num_layers=3
        )

        self.decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(d_model=300, nhead=5),
            num_layers=3
        )

        # Project encoder output (100 → 300)
        self.src_proj = nn.Linear(100, 300)

        self.fc_out = nn.Linear(300, 300)

    def generate_mask(self, sz):
        return torch.triu(torch.ones(sz, sz), diagonal=1).bool()

    def forward(self, src, tgt):
        src_len = src.shape[1]
        tgt_len = tgt.shape[1]

        src = src + get_positional_encoding(src_len, 100, src.device)
        tgt = tgt + get_positional_encoding(tgt_len, 300, tgt.device)

        src = src.permute(1, 0, 2)
        tgt = tgt.permute(1, 0, 2)

        memory = self.encoder(src)
        memory = self.src_proj(memory)

        tgt_mask = self.generate_mask(tgt_len).to(tgt.device)

        out = self.decoder(tgt, memory, tgt_mask=tgt_mask)
        out = self.fc_out(out)

        return out.permute(1, 0, 2)

model = TransformerMT().to(device)

/tmp/ipykernel_2382/566731089.py:8: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = nn.TransformerEncoder(


In [32]:
# criterion = nn.CrossEntropyLoss(ignore_index=token2idx['<pad>'])
# optimizer = optim.Adam(model.parameters(), lr=1e-4)
# from torch.amp import autocast, GradScaler
# scaler = GradScaler()
# epochs = 10
# train_losses, val_losses = [], []

# for epoch in range(epochs):
#     model.train()
#     epoch_loss = 0
#     for src, tgt in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
#         src, tgt = src.to(device), tgt.to(device)
#         tgt_input = tgt[:, :-1]
#         tgt_output = tgt[:, 1:]

#         optimizer.zero_grad()
#         with autocast('cuda'):
#             preds = model(src, tgt_input)
#             preds = preds.reshape(-1, preds.shape[-1])
#             tgt_output = tgt_output.reshape(-1)
#             loss = criterion(preds, tgt_output)

#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         epoch_loss += loss.item()

#     avg_train_loss = epoch_loss / len(train_loader)
#     train_losses.append(avg_train_loss)
#     print(f"Epoch {epoch+1} Train Loss: {avg_train_loss:.4f}")

#     # Validation loss
#     model.eval()
#     val_loss = 0
#     with torch.no_grad():
#         for src, tgt in val_loader:
#             src, tgt = src.to(device), tgt.to(device)
#             tgt_input = tgt[:, :-1]
#             tgt_output = tgt[:, 1:]
#             with autocast('cuda'):
#                 preds = model(src, tgt_input)
#                 preds = preds.reshape(-1, preds.shape[-1])
#                 tgt_output = tgt_output.reshape(-1)
#                 loss = criterion(preds, tgt_output)
#             val_loss += loss.item()
#     avg_val_loss = val_loss / len(val_loader)
#     val_losses.append(avg_val_loss)
#     print(f"Epoch {epoch+1} Val Loss: {avg_val_loss:.4f}")

In [ ]:
# =========================
# TRAINING
# =========================
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

epochs = 10
train_losses, val_losses = [], []

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for src, tgt in tqdm(train_loader):
        src, tgt = src.to(device), tgt.to(device)

        tgt_input = tgt[:, :-1, :]
        tgt_output = tgt[:, 1:, :]

        optimizer.zero_grad()

        preds = model(src, tgt_input)
        loss = criterion(preds, tgt_output)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    train_losses.append(train_loss)

    # Validation
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)

            tgt_input = tgt[:, :-1, :]
            tgt_output = tgt[:, 1:, :]

            preds = model(src, tgt_input)
            loss = criterion(preds, tgt_output)

            val_loss += loss.item()

    val_loss /= len(val_loader)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}: Train={train_loss:.4f}, Val={val_loss:.4f}")

# =========================
# VECTOR → WORD
# =========================
def vec_to_word(vec):
    return fasttext.similar_by_vector(vec, topn=1)[0][0]


100%|██████████| 4375/4375 [52:25<00:00,  1.39it/s]


Epoch 1: Train=0.0613, Val=0.0597


 64%|██████▍   | 2817/4375 [33:39<20:48,  1.25it/s]

In [ ]:
# detok = TreebankWordDetokenizer()


# def calculate_bleu_scores(model, dataloader, name="Test"):
#     model.eval()
#     all_references = []
#     all_predictions = []
#     examples = []

#     with torch.no_grad():
#         for src, tgt in tqdm(dataloader, desc=f"Evaluating {name}"):
#             src, tgt = src.to(device), tgt.to(device)
#             tgt_input = tgt[:, :-1]

#             # Get predictions
#             preds = model(src, tgt_input).argmax(-1)

#             for pred_seq, true_seq in zip(preds, tgt):
#                 pred_tokens = []
#                 for idx in pred_seq:
#                     token = idx2token.get(idx.item(), '<unk>')
#                     if token == '<eos>':
#                         break
#                     if token != '<pad>':
#                         pred_tokens.append(token)

#                 true_tokens = []
#                 for idx in true_seq:
#                     token = idx2token.get(idx.item(), '<unk>')
#                     if token in ['<pad>', '<sos>', '<eos>']:
#                         continue
#                     true_tokens.append(token)

#                 # Detokenize and replace underscores with spaces
#                 pred_sentence = detok.detokenize(pred_tokens).replace('▁', ' ').strip()
#                 true_sentence = detok.detokenize(true_tokens).replace('▁', ' ').strip()

#                 all_predictions.append(pred_sentence)
#                 all_references.append([true_sentence])
#                 examples.append((pred_sentence, true_sentence))

#     bleu = sacrebleu.corpus_bleu(all_predictions, all_references)

#     print(f"{name} Corpus BLEU score: {bleu.score:.2f}")
#     print(f"{name} BLEU Scores:")
#     for k in range(4):
#         print(f"BLEU@{k+1}: {bleu.precisions[k]:.2f}")

#     print(f"\n{name} Sample Predictions:")
#     for i in range(25):
#         print(f"Source: {detok.detokenize(nltk.word_tokenize(dataloader.dataset.sources[i]))}")
#         print(f"Target: {examples[i][1]}")
#         print(f"Predicted: {examples[i][0]}\n")

#     return bleu.score

In [ ]:
# =========================
# BLEU EVALUATION
# =========================
def evaluate(model, loader, name="Test"):
    model.eval()

    preds_all = []
    refs_all = []

    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)

            preds = model(src, tgt[:, :-1, :])

            for pred_seq, true_seq in zip(preds, tgt):
                pred_tokens = []
                true_tokens = []

                for vec in pred_seq:
                    word = vec_to_word(vec.cpu().numpy())
                    pred_tokens.append(word)

                for vec in true_seq:
                    word = vec_to_word(vec.cpu().numpy())
                    true_tokens.append(word)

                preds_all.append(" ".join(pred_tokens))
                refs_all.append([" ".join(true_tokens)])

    bleu = sacrebleu.corpus_bleu(preds_all, refs_all)

    print(f"\n{name} BLEU: {bleu.score:.2f}")
    print("BLEU-1:", bleu.precisions[0])
    print("BLEU-2:", bleu.precisions[1])
    print("BLEU-3:", bleu.precisions[2])
    print("BLEU-4:", bleu.precisions[3])

evaluate(model, test_loader, "Test")
evaluate(model, val_loader, "Validation")


In [ ]:

# =========================
# PLOT
# =========================
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.legend()
plt.title("Loss Curve")
plt.show()

In [ ]:

# test_bleu = calculate_bleu_scores(model, test_loader, "Test")
# val_bleu = calculate_bleu_scores(model, val_loader, "Validation")

# plt.figure(figsize=(10, 5))
# plt.plot(train_losses, label='Train Loss')
# plt.plot(val_losses, label='Validation Loss')
# plt.title('Losses over Epochs')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.legend()
# plt.grid()
# plt.show()

In [ ]:
# # Load model
# checkpoint = torch.load('transformer_mt_checkpoint.pth', map_location=device)
# model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])